# Распространение гауссовых пучков

Цель расчёта — численно смоделировать свободное распространение суперпозиции гауссовых пучков и проследить эволюцию распределения интенсивности.

## Постановка расчёта

Рассматривается когерентная суперпозиция трёх гауссовых пучков. Распространение выполняется методом углового спектра на равномерной двумерной сетке.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

from AO import GaussianBeam, Propagator

In [ ]:
N = 1024
dx = 1e-4
wavelength = 650e-9
L_screen = N * dx

phase_0 = np.zeros((N, N))

beam_0 = GaussianBeam(N=N, size_screen=L_screen, w0=L_screen / 30, x_0=L_screen / 5)
beam_1 = GaussianBeam(N=N, size_screen=L_screen, w0=L_screen / 30, x_0=-L_screen / 5)
beam_2 = GaussianBeam(N=N, size_screen=L_screen, w0=L_screen / 30, x_0=0.0, y_0=L_screen / 5)

E = (
    4 * beam_0.E * np.exp(1j * phase_0)
    + 4 * beam_1.E
    + 4 * beam_2.E
)

## Численное распространение

Поле последовательно переносится на одинаковые расстояния. На каждом шаге сохраняются интенсивность и фаза комплексной амплитуды.

In [ ]:
n_phase_screen = 80
dist = 5

intensity_all = [np.abs(E) ** 2]
phase_all = [phase_0]
mean_intensity = [intensity_all[0].mean()]

for _ in range(n_phase_screen):
    E = Propagator.propagate(N, dx, E, wavelength, dist, method="exact")
    intensity_all.append(np.abs(E) ** 2)
    phase_all.append(np.angle(E))
    mean_intensity.append(intensity_all[-1].mean())

## Распределение интенсивности

Показаны характерные плоскости наблюдения: начальная, промежуточные и конечная.

In [ ]:
extent_mm = [-L_screen * 500, L_screen * 500, -L_screen * 500, L_screen * 500]
frames = [0, n_phase_screen // 4, n_phase_screen // 2, n_phase_screen]

fig, axes = plt.subplots(2, 2, figsize=(11, 11), constrained_layout=True)

for ax, frame in zip(axes.ravel(), frames):
    im = ax.imshow(intensity_all[frame], cmap="hot", extent=extent_mm)
    im.set_clim(0, 1)
    ax.set_title(f"L = {frame * dist} м")
    ax.set_xlabel("x, мм")
    ax.set_ylabel("y, мм")
    fig.colorbar(im, ax=ax)

plt.show()

## Анимация распространения

Анимация отображает изменение поперечного распределения интенсивности при увеличении пройденного расстояния.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
im = ax.imshow(intensity_all[0], cmap="hot", extent=extent_mm)
im.set_clim(0, 1)
fig.colorbar(im, ax=ax, label="Интенсивность")
ax.set_xlabel("x, мм")
ax.set_ylabel("y, мм")

def update(frame):
    im.set_data(intensity_all[frame])
    ax.set_title(f"L = {frame * dist} м")
    return (im,)

animation = FuncAnimation(fig, update, frames=n_phase_screen + 1, interval=30, blit=True)
plt.close(fig)
HTML(animation.to_jshtml())